LionGuard 2 的核心技术路径是线性探测（Linear Probing），即冻结高性能的预训练嵌入模型，仅训练顶层的轻量化分类器 。这种方法允许模型在极低计算资源下（单 CPU 即可）实现极快的训练（不到 2 分钟）和推理（约 300 tokens/s） 。

以下是实现基于嵌入的多头序数分类器的具体步骤和代码示例：
1. 核心架构
    设计LionGuard 2 的架构分为两个主要部分：
    - 特征提取层（Frozen Backbone）： 使用 OpenAI 的 text-embedding-3-large（3072 维）作为特征提取器，处理多语言和新加坡式英语（Singlish）的语义特征 。
    - 多头序数头（Multi-head Ordinal Heads）： 针对每个风险类别（如仇恨言论、性内容）设置独立的分类头，并引入序数逻辑来区分风险等级 。
2. 序数分类逻辑（Ordinal Logic）LionGuard 2 引入了严重程度分级（Level 1 和 Level 2），并满足数学约束条件 $0 \leq p_2 \leq p_1 \leq 1$ 。其中：
    - $p_1$：代表该内容属于该类别（至少为 Level 1）的概率。
    - $p_2$：代表该内容属于严重违规（Level 2）的概率。

3. PyTorch 代码实现示例

以下代码展示了如何构建一个类似 LionGuard 2 的轻量化多头分类器：

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class OrdinalHead(nn.Module):
    """单类别的序数分类头 (Level 1 & Level 2)"""
    def __init__(self, input_dim):
        super(OrdinalHead, self).__init__()
        # 输出两个 logits：一个用于判断是否违规，一个用于判断是否严重违规
        self.fc = nn.Linear(input_dim, 2)
        
    def forward(self, x):
        logits = self.fc(x)
        # 确保 p2 <= p1 的一种常用结构化方法：
        # p1 = sigmoid(z1)
        # p2 = p1 * sigmoid(z2)
        p1 = torch.sigmoid(logits[:, 0:1])
        p2 = p1 * torch.sigmoid(logits[:, 1:2])
        return torch.cat([p1, p2], dim=1)

class LionGuard2LikeModel(nn.Module):
    """类似 LionGuard 2 的多头分类器架构"""
    def __init__(self, input_dim=3072, categories=None):
        super(LionGuard2LikeModel, self).__init__()
        if categories is None:
            # 定义 LionGuard 2 的核心风险类别 [5]
            categories = ['hate', 'sexual', 'self_harm', 'misconduct']
        
        # 为每个有等级的类别创建序数头
        self.heads = nn.ModuleDict({
            cat: OrdinalHead(input_dim) for cat in categories
        })
        
        # 对于没有等级的类别（如 Insults, Violence），使用普通的二元头 [5]
        self.binary_heads = nn.ModuleDict({
            'insults': nn.Linear(input_dim, 1),
            'physical_violence': nn.Linear(input_dim, 1)
        })

    def forward(self, embeddings):
        results = {}
        # 处理带等级的头
        for cat, head in self.heads.items():
            results[cat] = head(embeddings)
        
        # 处理二元头
        for cat, head in self.binary_heads.items():
            results[cat] = torch.sigmoid(head(embeddings))
            
        return results

# 初始化模型 (OpenAI text-embedding-3-large 的维度是 3072)
model = LionGuard2LikeModel(input_dim=3072)
print(model)

LionGuard2LikeModel(
  (heads): ModuleDict(
    (hate): OrdinalHead(
      (fc): Linear(in_features=3072, out_features=2, bias=True)
    )
    (sexual): OrdinalHead(
      (fc): Linear(in_features=3072, out_features=2, bias=True)
    )
    (self_harm): OrdinalHead(
      (fc): Linear(in_features=3072, out_features=2, bias=True)
    )
    (misconduct): OrdinalHead(
      (fc): Linear(in_features=3072, out_features=2, bias=True)
    )
  )
  (binary_heads): ModuleDict(
    (insults): Linear(in_features=3072, out_features=1, bias=True)
    (physical_violence): Linear(in_features=3072, out_features=1, bias=True)
  )
)


4. 训练方案与数据集建议要达到 LionGuard 2 的效果，训练过程应关注以下几点：
- 数据集规模与构成： 准备约 2.6 万条样本的紧凑数据集，包含本地论坛（如 HardwareZone）的真实评论、由大模型重写的合成查询以及经过筛选的开源英语毒性数据 。
- 半监督标注： 利用 Gemini 2.0 Flash 或 Claude 3.5 Haiku 等先进模型对本地数据进行自动标注 。
- 训练目标函数：
    - 对于序数头，可以使用均方误差（MSE）或二元交叉熵（BCE）的组合损失。
    - 由于特征提取层是冻结的，仅训练全连接层权重，因此在单 CPU 上即可完成快速收敛 。
- 数据增强： 加入拼写错误、标点异常、大小写不一等噪声数据，以提高模型在真实非正式社交文本中的鲁棒性 。

5. 推理流程
    1. 文本嵌入： 调用 OpenAI API 或本地多语言嵌入模型提取文本向量。
    2. 分类： 将向量输入上述 PyTorch 模型。
    3. 阈值判定： 根据输出的概率值（如 $>0.5$）判定风险类别和等级。例如，如果 hate_p1 > 0.5 且 hate_p2 > 0.5，则判定为“严重仇恨言论”。

完全可以使用开源模型替代 OpenAI。事实上，LionGuard 2 在研发过程中也对比了多种开源嵌入模型，如 BGE-M3 和 Qwen3-Embedding 。


1. 推荐的开源替代方案
如果你希望摆脱对 OpenAI API 的依赖，以下模型是针对多语言及新加坡语境（Singlish）的优秀选择：
- BGE-M3 (BAAI): 支持 100 多种语言，具有 1024 维嵌入空间。它在处理低资源语言和多功能检索方面表现稳健，是目前最流行的开源基准之一。

- Qwen3-Embedding-0.6B: 阿里巴巴推出的轻量化模型。研究表明，它在多语言评估（MMTEB）中比 BGE-M3 有约 7.9% 的相对提升，且同样具备 1024 维的输出。

- SEA-LION 系列 (AI Singapore): 虽然专门的 SEA-Embedding 模型仍在“即将推出”阶段，但 AI Singapore 的模型在东南亚语义理解上具有天然优势。

2. 实现代码示例（基于 Hugging Face）
你可以使用 sentence-transformers 库轻松替换 OpenAI 的嵌入提取部分。以下是一个使用 BGE-M3 作为后端的完整训练/推理架构示例：

In [5]:
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer

# 1. 加载开源嵌入模型 (代替 OpenAI)
# BGE-M3 的维度通常是 1024
embed_model = SentenceTransformer('BAAI/bge-m3')
input_dim = 1024 

# 2. 定义多头序数分类器 (Ordinal Classifier)
class MultiHeadOrdinalClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        # 每个风险类别对应一个序数头
        self.heads = nn.ModuleDict({
            'hate': nn.Linear(input_dim, 2),    # 输出 z1, z2
            'sexual': nn.Linear(input_dim, 2),
            'violence': nn.Linear(input_dim, 1) # 某些类可以是二元分类
        })

    def forward(self, x):
        outputs = {}
        # 序数逻辑实现: p1 = sigmoid(z1), p2 = p1 * sigmoid(z2)
        # 从而严格保证数学约束: $0 \leq p_2 \leq p_1 \leq 1$
        for name, head in self.heads.items():
            logits = head(x)
            if logits.shape[1] == 2:
                p1 = torch.sigmoid(logits[:, 0:1])
                p2 = p1 * torch.sigmoid(logits[:, 1:2])
                outputs[name] = torch.cat([p1, p2], dim=1)
            else:
                outputs[name] = torch.sigmoid(logits)
        return outputs

# 3. 推理流程
def predict(text, classifier):
    # 提取特征 (确保不更新嵌入模型的权重，即 Linear Probing)
    with torch.no_grad():
        embeddings = embed_model.encode([text], convert_to_tensor=True)
    
    # 分类判定
    scores = classifier(embeddings)
    return scores

# 初始化
# classifier = MultiHeadOrdinalClassifier(input_dim=input_dim, num_classes=3)

KeyboardInterrupt: 

In [4]:
import os
import numpy as np
from transformers import AutoModel
from openai import OpenAI

# Load model directly from HF
model = AutoModel.from_pretrained(
    "govtech/lionguard-2", 
    trust_remote_code=True
    )

# Get OpenAI embeddings (users to input their own OpenAI API key)
client = OpenAI(api_key="sk-VbQ1X7SP3rrtG6btyDMm9wFHdNpJGfCCHuwM6h7oWJS3z2xi")
print(client)
response = client.embeddings.create(
    input="Hello, world!", # users to input their own text
    model="text-embedding-3-large",
    dimensions=3072 # dimensions of the embedding
    )
embeddings = np.array([data.embedding for data in response.data])
print(embeddings)
# Run LionGuard 2
results = model.predict(embeddings)


'[Errno 101] Network is unreachable' thrown while requesting HEAD https://huggingface.co/govtech/lionguard-2/resolve/main/config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.